In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from IPython.display import display
# import ace_tools as tools

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

from sklearn.feature_selection import RFECV
from sklearn.pipeline import make_pipeline

import seaborn as sns

from imblearn.over_sampling import BorderlineSMOTE


### import data

In [ ]:
df = pd.read_excel('Compost_Data.xlsx')

### class determination

In [ ]:
def classify_maturity(row):
  if row["GI"] < 70:
    return "Immature"
  elif 70<= row["GI"] <= 80:
    return "Moderately Mature"
  else: 
    return "Mature"

df["Maturity_Class"] = df.apply(classify_maturity, axis=1)

### normalize data from 0 to 1

In [ ]:
def scalled_data(df):
  df_new = df.drop(columns=["Maturity_Score", "GI","T Value", "Maturity_Class"])
  numerical_columns = df_new.select_dtypes(include=["float64", "int64"]).columns.tolist()
  scaler = MinMaxScaler(feature_range=(0, 1))
  df_norm = pd.DataFrame(scaler.fit_transform(df[numerical_columns]), columns=numerical_columns)
  df_scaled = pd.concat([df_norm.reset_index(drop=True), df['Maturity_Class']],axis=1)
  return df_scaled

In [ ]:
df_scaled =scalled_data(df)

In [ ]:
M_counts= df_scaled["Maturity_Class"].value_counts()
M_counts

In [ ]:
stats_by_class = df.groupby("Maturity_Class").agg(["mean", "std"])
stats_by_class
stats_by_class.to_excel('class_description1.xlsx')

### spider chart

In [ ]:
def spider_chart(df):
    
  df["Maturity_Class"] = df["Maturity_Class"].str.lower().str.strip()
  numerical_columns = df.select_dtypes(include=["float64", "int64"]).columns.tolist()
  grouped_data = df.groupby("Maturity_Class")[numerical_columns].mean()


  custom_order =["immature", "moderately mature", "mature"]
  grouped_data = grouped_data.loc[custom_order]

  labels = numerical_columns
  num_vars = len(labels)
  angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
  angles.append(angles[0])
  fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

  max_val = grouped_data.max().max()

  for maturity_class, values in grouped_data.iterrows():
        values = values.tolist()
        values.append(values[0])  
        ax.plot(angles, values, label=maturity_class, linewidth=2, linestyle="solid")
        ax.fill(angles, values, alpha=0.25)

  ax.set_xticks(angles[:-1])
  ax.set_xticklabels(labels, fontsize=20)
  ax.set_yticks([])
  ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=20)
  plt.show()

In [ ]:
plot = spider_chart(df_scaled)

In [ ]:
x_data=df_scaled.drop(columns=["Maturity_Class"])
y_data=df_scaled["Maturity_Class"]
blsmote = BorderlineSMOTE(sampling_strategy='not majority', kind='borderline-1', random_state=42)
X_data_blsmote, y_data_blsmote = blsmote.fit_resample(x_data, y_data)


print("Class distribution AFTER Borderline-SMOTE:")
print(y_data_blsmote.value_counts())

### data split

In [ ]:
def data_split(x_data, y_data):
    x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.20, random_state=42)
    return x_train, x_test, y_train, y_test

x_train, x_test, y_train, y_test = data_split(X_data_blsmote, y_data_blsmote)


### grid hyperparameter search

In [ ]:
### GB model

x_train, y_train = make_classification(n_samples=542, n_features=11, random_state=42)
gb = GradientBoostingClassifier()
param_grid = {
    'n_estimators': [50, 100, 200], 
    'learning_rate': [0.01, 0.05, 0.1, 0.2],  
    'max_depth': [2, 3, 4, 5],  
    'min_samples_split': [2, 5, 10],  
    'min_samples_leaf': [1, 3, 5],  
    'subsample': [0.6, 0.8, 1.0],  
    'max_features': ['sqrt', 'log2', None]  
}
grid_search = GridSearchCV(gb, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(x_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)

In [ ]:
# KNN
x_train, y_train = make_classification(n_samples=429, n_features=11, random_state=42)
knn = KNeighborsClassifier()
param_grid = {
    'n_neighbors': list(range(1, 31)),  
    'weights': ['uniform', 'distance'],  
    'metric': ['euclidean', 'manhattan', 'minkowski'],  
    'p': [1, 2] 
}
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(x_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)

In [ ]:
### SVM
svm = SVC()
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],  
    'kernel': ['linear', 'rbf', 'poly'],  
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],  
    'degree': [2, 3, 4]  
}
grid_search = GridSearchCV(svm, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(x_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)

In [ ]:
### DT
dt = DecisionTreeClassifier(random_state=42)
param_grid = {
    'max_depth': [3, 5, 10, None],  
    'min_samples_split': [2, 5, 10, 20],  
    'min_samples_leaf': [1, 3, 5, 10],  
    'max_features': ['sqrt', 'log2', None],  
    'criterion': ['gini', 'entropy']  
}
grid_search = GridSearchCV(dt, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(x_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)

In [ ]:
### RF
rf = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200, 300],  
    'max_depth': [3, 5, 10, None],  
    'min_samples_split': [2, 5, 10],  
    'min_samples_leaf': [1, 3, 5],  
    'max_features': ['sqrt', 'log2', None], 
    'bootstrap': [True, False]
}
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(x_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)

In [ ]:
model_KNN = KNeighborsClassifier(metric='manhattan', n_neighbors=10, p=1, weights='uniform')
model_SVM =  SVC(C=100, degree=2, gamma=0.01, kernel='rbf')
model_GB = GradientBoostingClassifier(learning_rate=0.05, max_depth=5, max_features=None, min_samples_leaf=3, min_samples_split=10, n_estimators=200, subsample=0.8)
model_DT = DecisionTreeClassifier(criterion='entropy', max_depth=5, max_features=None, min_samples_leaf=1, min_samples_split=20)
model_RF = RandomForestClassifier (bootstrap=True, max_depth=10, max_features='sqrt', min_samples_leaf=1, min_samples_split=2, n_estimators=200)

### evaluation metrics

In [ ]:
def eval_model(y_true, y_pred):
  class_labels = ["Mature", "Moderately Mature", "Immature"]
  precision = precision_score(y_true, y_pred, average='weighted')
  recall = recall_score(y_true, y_pred, average='weighted')
  f1 = f1_score(y_true, y_pred, average='weighted')
  accuracy = accuracy_score(y_true, y_pred)
  # cm = pd.DataFrame(confusion_matrix(y_true, y_pred), index=class_labels, columns=class_labels)
  
  return precision, recall, f1, accuracy

### cross validation

In [ ]:
def cv_shuffle(x_data, y_data, model):
  cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
  fold_scores=[]
  fold =1
  for train_idx, val_idx in cv.split(x_data, y_data):
      X_train, X_val = x_data.iloc[train_idx], x_data.iloc[val_idx]  
      y_train, y_val = y_data.iloc[train_idx], y_data.iloc[val_idx]

      model.fit(X_train, y_train)
      y_pred = model.predict(X_val)
      accuracy, precision, recall, f1 = eval_model(y_val, y_pred)
      fold_scores.append([fold, accuracy, precision, recall, f1])
      fold += 1
  fold_scores_df = pd.DataFrame(fold_scores, columns=["Fold", "Accuracy", "Precision", "Recall", "F1 Score"])
  return fold_scores_df

### confusion metrics

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_labels):
    class_labels = [
        lbl if lbl != "moderately mature" else "moderately\nmature"
        for lbl in class_labels
    ]
    cm = confusion_matrix(y_true, y_pred)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    plt.figure(figsize=(6,5))
    ax = sns.heatmap(
        cm_percent,
        annot=True,
        fmt=".0f",
        cmap="Blues",
        xticklabels=class_labels,
        yticklabels=class_labels,
        annot_kws={"fontsize": 20},
        vmin=0, vmax=100,
        cbar_kws={"ticks": [0,20,40,60,80,100]}
    )
    ax.set_xticklabels(class_labels, fontsize=16)
    ax.set_yticklabels(class_labels, fontsize=16)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=18)

    plt.xlabel("Predicted label", fontsize=18)
    plt.ylabel("True label", fontsize=18)

    plt.tight_layout()
    plt.show()

### evaluate on training set

In [ ]:
def evaluate_models(x_data, y_data):
    models = {
        "KNN": model_KNN,
        "SVM": model_SVM,
        "GB": model_GB,
        "DT": model_DT,
        "RF": model_RF
    }

    results = []

    for name, model in models.items():
        cv_results = cv_shuffle(x_data, y_data, model)

        avg_accuracy = cv_results["Accuracy"].mean()
        avg_precision = cv_results["Precision"].mean()
        avg_recall = cv_results["Recall"].mean()
        avg_f1 = cv_results["F1 Score"].mean()

        results.append({
            "Model": name,
            "Accuracy": avg_accuracy,
            "Precision": avg_precision,
            "Recall": avg_recall,
            "F1 Score": avg_f1
        })

    results_df = pd.DataFrame(results)

    return results_df

In [ ]:
mean_score= evaluate_models(x_train, y_train)
mean_score

### test model on test set

In [ ]:
def test_models_on_testset(X_train, y_train, X_test, y_test):
    models = {
        "KNN": model_KNN,
        "SVM": model_SVM,
        "GB": model_GB,
        "DT": model_DT,
        "RF": model_RF
    }

    results = []

    class_labels = ["immature", "moderately mature", "mature"]

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, average="weighted"),
            "Recall": recall_score(y_test, y_pred, average="weighted"),
            "F1-score": f1_score(y_test, y_pred, average="weighted"),
        })

        print(f"\n{name} Confusion Matrix:")
        plot_confusion_matrix(y_test, y_pred, class_labels)

    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values(by="F1-score", ascending=False).reset_index(drop=True)

    return df_results

In [ ]:
metric_score = test_models_on_testset(x_train, y_train, x_test, y_test)
metric_score

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

def recursive_feature_addition(x_data, y_data, model):
    scaler = StandardScaler()
    x_data_scaled = pd.DataFrame(scaler.fit_transform(x_data), columns=x_data.columns)

    x_train, x_test, y_train, y_test = train_test_split(x_data_scaled, y_data, test_size=0.2, random_state=42, stratify=y_data)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    score_data = []
    selected_features = []
    remaining_features = list(x_train.columns)

    while len(remaining_features) > 0:
        best_feature = None
        best_score = 0

        for feature in remaining_features:
            temp_features = selected_features + [feature]
            scores = cross_val_score(model, x_train[temp_features], y_train, cv=skf, scoring="accuracy", n_jobs=-1)
            mean_score = scores.mean()

            if mean_score > best_score:
                best_score = mean_score
                best_feature = feature

        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        score_data.append([len(selected_features), best_score, best_feature])

    scores_df = pd.DataFrame(score_data, columns=["Number of Features", "Cross-Validation Accuracy", "Added Feature"])

    plt.figure(figsize=(8, 6))
    plt.plot(scores_df["Number of Features"], scores_df["Cross-Validation Accuracy"], marker="o", linestyle="--")
    plt.xlabel("Number of Features", fontsize=12)
    plt.ylabel("Cross-Validation Accuracy", fontsize=12)
    plt.title("Recursive Feature Addition", fontsize=14)
    plt.grid(True)
    plt.show()

    return scores_df


In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

def recursive_feature_addition(x_data, y_data, model):
    scaler = StandardScaler()
    x_data_scaled = pd.DataFrame(scaler.fit_transform(x_data), columns=x_data.columns)

    # x_train, x_test, y_train, y_test = train_test_split(x_data_scaled, y_data, test_size=0.2, random_state=42, stratify=y_data)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    score_data = []
    selected_features = []
    remaining_features = list(x_train.columns)

    while len(remaining_features) > 0:
        best_feature = None
        best_score = 0

        for feature in remaining_features:
            temp_features = selected_features + [feature]
            scores = cross_val_score(model, x_data_scaled[temp_features], y_data, cv=skf, scoring="accuracy", n_jobs=-1)
            mean_score = scores.mean()

            if mean_score > best_score:
                best_score = mean_score
                best_feature = feature

        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        score_data.append([len(selected_features), best_score, best_feature])

    scores_df = pd.DataFrame(score_data, columns=["Number of Features", "Cross-Validation Accuracy", "Added Feature"])

    plt.figure(figsize=(8, 6))
    plt.plot(scores_df["Number of Features"], scores_df["Cross-Validation Accuracy"], marker="o", linestyle="--")
    plt.xlabel("Number of Features", fontsize=12)
    plt.ylabel("Cross-Validation Accuracy", fontsize=12)
    plt.title("Recursive Feature Addition", fontsize=14)
    plt.grid(True)
    plt.show()

    return scores_df


In [ ]:
df_score_KNN = recursive_feature_addition(X_data_blsmote, y_data_blsmote, model_KNN)
df_score_KNN

In [ ]:
df_score_SVM = recursive_feature_addition(X_data_blsmote, y_data_blsmote, model_SVM)
df_score_SVM

In [ ]:
df_score_GB = recursive_feature_addition(X_data_blsmote, y_data_blsmote, model_GB)
df_score_GB

In [ ]:
df_score_DT = recursive_feature_addition(X_data_blsmote, y_data_blsmote, model_DT)
df_score_DT

In [ ]:
df_score_RF = recursive_feature_addition(X_data_blsmote, y_data_blsmote, model_RF)
df_score_RF

In [ ]:
df_RFA = pd.concat([df_score_KNN, df_score_SVM, df_score_GB, df_score_DT, df_score_RF], axis=1).drop(columns=["Number of Features"])
df_RFA.to_excel('cross validation accuracy.xlsx')

In [ ]:
x = df_score_KNN["Number of Features"]
y1 = df_score_KNN["Cross-Validation Accuracy"]
y2 = df_score_SVM["Cross-Validation Accuracy"]
y3 = df_score_GB["Cross-Validation Accuracy"]
y4 = df_score_DT["Cross-Validation Accuracy"]
y5 = df_score_RF["Cross-Validation Accuracy"]

plt.figure(figsize=(12, 6))

plt.plot(x, y1, label='KNN model', linestyle='-', marker='o')
plt.plot(x, y2, label='SVM model', linestyle='-', marker='s')
plt.plot(x, y4, label='DT model', linestyle='-', marker='*')
plt.plot(x, y3, label='GB model', linestyle='-', marker='^')
plt.plot(x, y5, label='RF model', linestyle='-', marker='o')


plt.xlabel('Number of Added Features', fontsize=20)
plt.ylabel('Cross Validation Accuracy', fontsize=20)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.xlim(0,12)

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=5, fontsize=16)
plt.grid(True)
plt.show()